# P1 smoke test — 20 items, English, T4 x2

This notebook runs the **mandatory 20-item smoke test** from section 9 of the P1
brief. It does not run the five-language experiment. Nothing here is a P1 cell:
a limited run is tagged `smoke` and `build_tidy` would reject it.

Run order is deliberate.

1. **Settings** — Accelerator **GPU T4 x2**, Internet **ON**.
2. **Probe** — a P100 cannot run INT8 or NF4; cell 3 stops you.
3. **Verify the frozen contracts** — P0 freeze, P1 split manifest, full pytest.
4. **Smoke** — train 20 English items, merge, load FP16/NF4/INT8, score.
5. **Read the report** — all eight checks must pass before anything larger runs.

Everything that defines the experiment lives in `configs/experiment.yaml` and
the two frozen manifests. Nothing is configured here.

In [ ]:
# 1. Get the code. Push D:\quantlang to this repo first.
REPO_URL = "https://github.com/fairuz-anadi/quantization.git"
REF      = "main"          # pin to a commit SHA once the run is the real one

import os, subprocess, sys
WORK = "/kaggle/working"
SRC  = f"{WORK}/quantlang"

if not os.path.exists(SRC):
    subprocess.run(["git", "clone", "--depth", "1", "-b", REF, REPO_URL, SRC], check=True)
print(subprocess.run(["git", "-C", SRC, "rev-parse", "HEAD"],
                     capture_output=True, text=True).stdout.strip())

os.chdir(SRC)
sys.path.insert(0, SRC)

In [ ]:
# 2. Dependencies. Kaggle's torch is CUDA-matched -- never reinstall it.
!pip install -q -U "transformers>=4.45" "bitsandbytes>=0.43" "peft>=0.13"     accelerate datasets pyyaml

In [ ]:
# 3. Environment probe. STOP HERE if this exits non-zero.
!python scripts/probe_env.py --outdir /kaggle/working

In [ ]:
# 4. The frozen contracts must hold before any GPU time is spent.
#    - P0 must be byte-identical to what produced the published results
#    - the P1 split must reproduce exactly from its seed and revision
!python scripts/freeze_p0.py
!python -m pytest -q

### Check before continuing

`pytest` must be green and `freeze_p0.py` must report the P0 freeze intact. The
30 unregistered P0 raw-provenance files are a known pre-existing gap, not a
failure.

The split check below re-derives all five languages from the pinned dataset
revision and compares them against the frozen digests. It downloads the corpus,
so it takes a few minutes.

In [ ]:
# 5. Re-derive the P1 split and verify it against the frozen manifest.
!python scripts/build_p1_splits.py --check

In [ ]:
# 6. The smoke test: train 20 English items, merge, load at all three
#    precisions, score BELEBELE and held-out items through the P0 evaluator.
#    Returns non-zero if any of the eight checks fails.
!python scripts/run_p1_smoke.py --outdir /kaggle/working/p1_smoke --lang eng_Latn

In [ ]:
# 7. Read the report.
import json
r = json.load(open("/kaggle/working/p1_smoke/p1_smoke_report.json", encoding="utf-8"))

for name, check in sorted(r["checks"].items()):
    print(f"[{'PASS' if check['pass'] else 'FAIL'}] {name}")

ft = r["finetune"]
print("
--- fine-tune ---")
print(f"trainable params : {ft['parameter_counts']['trainable_parameters']:,}"
      f" ({ft['parameter_counts']['trainable_percent']:.4f}%)")
print(f"base params      : {ft['parameter_counts']['base_parameters']:,}")
print(f"loss first/last  : {ft['training']['loss_first_decile_mean']:.4f}"
      f" -> {ft['training']['loss_last_decile_mean']:.4f}")
print(f"train seconds    : {ft['training']['train_seconds']:.1f}")
print(f"adapter          : {ft['adapter']['total_bytes'] / 1024**2:.1f} MB")
print(f"merged ckpt      : {ft['merged_checkpoint']['total_gb']:.2f} GB")

print("
--- precision ---")
for prec, v in r["per_precision"].items():
    print(f"{prec:<14} layers={v['layer_counts']}"
          f" peak={v['peak_memory_allocated_gb']:.2f}GB"
          f" belebele={v['belebele_accuracy']:.2f}"
          f" heldout={v['heldout_accuracy']:.2f}"
          f" letters={v['distinct_predicted_letters']}")
print(f"
base FP16 belebele accuracy: {r['base_fp16']['belebele_accuracy']:.2f}")
print(f"
ALL CHECKS PASSED: {r['all_checks_passed']}")

In [ ]:
# 8. Package the smoke artefacts for download.
import shutil, os
shutil.make_archive("/kaggle/working/p1_smoke_artifacts", "zip",
                    "/kaggle/working/p1_smoke")
print(sorted(os.listdir("/kaggle/working")))

### Then, locally

```bash
kaggle kernels output <user>/<kernel-slug> -p results/P1/metadata/
```

Only once every check reads PASS does the five-language run become worth
starting. If any check fails, stop and report it — the experimental design is
frozen and is not adjusted to make a check pass.